In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================

dbutils.widgets.text("path", "")

config_path = dbutils.widgets.get("path")


# ============================================================
# 1. IMPORTS
# ============================================================

import json

import pyspark.sql.functions as F

from common_utils.logging import get_logger
from common_utils.generic_functions import safe_cast


# ============================================================
# 2. LOGGER
# ============================================================

logger = get_logger("dim_customer_gold")


# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",
    config_path
)

with open(config_path, "r") as f:
    config = json.load(f)

logger.info(
    "Gold configuration loaded successfully"
)


# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================

source_config = config["source"]
target_config = config["target"]

source_name = source_config["source_name"]

source_table = source_config["source_table"]

target_table = target_config["target_table"]


logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)


# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading customers from %s",
    source_name,
    source_table
)

customers_df = spark.read.table(
    source_table
)

logger.info(
    "[%s] Customers table loaded successfully",
    source_name
)


# ============================================================
# 6. TRANSFORM CUSTOMER
# ============================================================

def transform_customer(df, config):

    logger.info(
        "[%s] Starting dim_customer transformation",
        source_name
    )

    # --------------------------------------------------------
    # Select required columns
    # --------------------------------------------------------

    df = df.select(
        "customer_id",
        "first_name",
        "last_name",
        "loyalty_segment",
        "valid_from",
        "last_update_ts"
    )


    # --------------------------------------------------------
    # Clean names
    # --------------------------------------------------------

    df = (
        df
        .withColumn(
            "first_name",
            F.trim(F.col("first_name"))
        )
        .withColumn(
            "last_name",
            F.trim(F.col("last_name"))
        )
    )


    # --------------------------------------------------------
    # Create full name
    # --------------------------------------------------------

    df = df.withColumn(
        "full_name",
        F.trim(
            F.concat_ws(
                " ",
                F.col("first_name"),
                F.col("last_name")
            )
        )
    )


    # --------------------------------------------------------
    # Derive customer type
    #
    # Keep business meaning configurable rather than assuming
    # what loyalty segment 0/1/2 means.
    # --------------------------------------------------------

    df = df.withColumn(
        "customer_type",
        F.when(
            F.col("loyalty_segment").isNull(),
            F.lit(
                config["transform"]["default_customer_type"]
            )
        )
        .otherwise(
            F.concat(
                F.lit("Segment_"),
                F.col("loyalty_segment").cast("string")
            )
        )
    )


    # --------------------------------------------------------
    # Created date
    # --------------------------------------------------------

    df = df.withColumn(
        "created_date",
        F.to_date(
            F.from_unixtime(
                F.col("valid_from").cast("long")
            )
        )
    )


    # --------------------------------------------------------
    # Last updated
    # --------------------------------------------------------

    df = df.withColumn(
        "last_updated",
        F.to_timestamp(
            F.col("last_update_ts")
        )
    )


    # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    df = safe_cast(
        df,
        config.get("cast_columns", {})
    )


    # --------------------------------------------------------
    # Select final columns
    # --------------------------------------------------------

    df = df.select(
        *config["select_columns"]
    )


    # --------------------------------------------------------
    # Remove duplicate customers
    # --------------------------------------------------------

    df = (
        df
        .orderBy(
            F.col("last_updated").desc()
        )
        .dropDuplicates(
            ["customer_id"]
        )
    )


    logger.info(
        "[%s] dim_customer transformation completed",
        source_name
    )

    return df


# ============================================================
# 7. CREATE DIM CUSTOMER
# ============================================================

logger.info(
    "[%s] Starting dim_customer creation",
    source_name
)

dim_customer = transform_customer(
    customers_df,
    config
)


# ============================================================
# 8. VALIDATE DATA
# ============================================================

logger.info(
    "[%s] Validating dim_customer",
    source_name
)

record_count = dim_customer.count()

logger.info(
    "[%s] dim_customer record count: %s",
    source_name,
    record_count
)

if record_count == 0:

    raise ValueError(
        "dim_customer contains zero records"
    )


duplicate_count = (
    dim_customer
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)


if duplicate_count > 0:

    raise ValueError(
        "Duplicate customer_id detected in dim_customer"
    )


# ============================================================
# 9. DISPLAY
# ============================================================

dim_customer.show()


# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================

logger.info(
    "[%s] Creating Gold schema if required",
    source_name
)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS
    retaildataplatform.gold
""")


# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_customer to %s",
    source_name,
    target_table
)

(
    dim_customer
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .saveAsTable(target_table)
)


logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)


# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_customer record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_customer pipeline completed successfully",
    source_name
)